In [4]:
import os
import numpy as np
import pandas as pd

raw_data_dir = "../data/final_test/raw_data"
brightfield_dir = "../data/final_test/brightfield_frames"
gfp_dir = "../data/final_test/gfp_frames"
filament_dir = "../data/final_test/filament_mask"
cell_mask_dir = "../data/final_test/cellpose_mask"
output_dir = "../data/final_test/results"

Dont put in final, just for testing

In [5]:
# Preview pillar removal, based on example dataset pillar values

import os
import numpy as np
import pandas as pd

# Load
if "cell_tracks_with_filaments_and_ids_df" in globals():
    df = cell_tracks_with_filaments_and_ids_df.copy()
else:
    csv_path = os.path.join(output_dir, "cell_tracks_per_frame_with_filaments_and_ids.csv")
    df = pd.read_csv(csv_path)

# Remove old helper columns if rerunning
helper_cols = [
    "is_pillar", "n_frames", "track_motion_std",
    "median_eccentricity", "median_aspect_ratio", "median_solidity"
]
df = df.drop(columns=[c for c in helper_cols if c in df.columns], errors="ignore")

# Required columns
required_cols = [
    "movie", "cell_ID", "frame",
    "centroid_y", "centroid_x",
    "eccentricity", "major_axis_length", "minor_axis_length", "solidity"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Clean numeric columns
work = df.copy()
for col in [
    "cell_ID", "frame", "centroid_y", "centroid_x",
    "eccentricity", "major_axis_length", "minor_axis_length", "solidity"
]:
    work[col] = pd.to_numeric(work[col], errors="coerce")

if "area" in work.columns:
    work["area"] = pd.to_numeric(work["area"], errors="coerce")

work = work.dropna(subset=["cell_ID"]).copy()
work["cell_ID"] = work["cell_ID"].astype(int)

work["aspect_ratio"] = work["major_axis_length"] / work["minor_axis_length"].replace(0, np.nan)

# Per-track summary
agg_dict = {
    "frame": "nunique",
    "centroid_y": ["std", "median"],
    "centroid_x": ["std", "median"],
    "eccentricity": "median",
    "solidity": "median",
    "aspect_ratio": "median",
}
if "area" in work.columns:
    agg_dict["area"] = "median"

track_summary = work.groupby(["movie", "cell_ID"]).agg(agg_dict)
track_summary.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c
    for c in track_summary.columns
]
track_summary = track_summary.reset_index().rename(columns={
    "frame_nunique": "n_frames",
    "centroid_y_std": "centroid_y_std",
    "centroid_x_std": "centroid_x_std",
    "centroid_y_median": "median_centroid_y",
    "centroid_x_median": "median_centroid_x",
    "eccentricity_median": "median_eccentricity",
    "solidity_median": "median_solidity",
    "aspect_ratio_median": "median_aspect_ratio",
    "area_median": "median_area"
})

track_summary["centroid_y_std"] = track_summary["centroid_y_std"].fillna(0)
track_summary["centroid_x_std"] = track_summary["centroid_x_std"].fillna(0)
track_summary["track_motion_std"] = np.sqrt(
    track_summary["centroid_y_std"]**2 + track_summary["centroid_x_std"]**2
)

# Versions tuned closer to your pillars (IDs 4 and 5 in example)
pillar_versions = {
    "v1_conservative": {
        "MIN_TRACK_LEN": 4,
        "MAX_MOTION_STD": 2.0,
        "MIN_ECC": 0.75,
        "MIN_ASPECT": 1.45,
        "MIN_SOLIDITY": 0.95,
    },
    "v2_balanced": {
        "MIN_TRACK_LEN": 4,
        "MAX_MOTION_STD": 2.5,
        "MIN_ECC": 0.72,
        "MIN_ASPECT": 1.40,
        "MIN_SOLIDITY": 0.95,
    },
    "v3_aggressive": {
        "MIN_TRACK_LEN": 3,
        "MAX_MOTION_STD": 3.0,
        "MIN_ECC": 0.68,
        "MIN_ASPECT": 1.35,
        "MIN_SOLIDITY": 0.94,
    },
}

preview_results = []
preview_filtered_dfs = {}
preview_track_tables = {}

for version_name, p in pillar_versions.items():
    ts = track_summary.copy()

    is_pillar = (
        (ts["n_frames"] >= p["MIN_TRACK_LEN"]) &
        (ts["track_motion_std"] <= p["MAX_MOTION_STD"]) &
        (ts["median_eccentricity"] >= p["MIN_ECC"]) &
        (ts["median_aspect_ratio"] >= p["MIN_ASPECT"]) &
        (ts["median_solidity"] >= p["MIN_SOLIDITY"])
    )

    ts["is_pillar"] = is_pillar

    df_labeled = df.merge(
        ts[[
            "movie", "cell_ID", "is_pillar", "n_frames",
            "track_motion_std", "median_eccentricity",
            "median_aspect_ratio", "median_solidity"
        ]],
        on=["movie", "cell_ID"],
        how="left"
    )

    df_labeled["is_pillar"] = df_labeled["is_pillar"].fillna(False)
    filtered_df = df_labeled.loc[~df_labeled["is_pillar"]].copy()

    preview_filtered_dfs[version_name] = filtered_df
    preview_track_tables[version_name] = ts.loc[ts["is_pillar"]].copy()

    preview_results.append({
        "version": version_name,
        "pillar_tracks_removed": int(ts["is_pillar"].sum()),
        "rows_removed": int(df_labeled["is_pillar"].sum()),
        "rows_remaining": int((~df_labeled["is_pillar"]).sum()),
        "min_track_len": p["MIN_TRACK_LEN"],
        "max_motion_std": p["MAX_MOTION_STD"],
        "min_ecc": p["MIN_ECC"],
        "min_aspect": p["MIN_ASPECT"],
        "min_solidity": p["MIN_SOLIDITY"],
    })

preview_summary_df = pd.DataFrame(preview_results).sort_values("version").reset_index(drop=True)

print("Preview summary:")
display(preview_summary_df)

for version_name in pillar_versions:
    print("\n" + "="*100)
    print(version_name)
    print("="*100)
    removed = preview_track_tables[version_name].copy()
    if removed.empty:
        print("No pillar candidates found.")
    else:
        display(
            removed[[
                "movie", "cell_ID", "n_frames", "track_motion_std",
                "median_eccentricity", "median_aspect_ratio", "median_solidity"
            ]]
            .sort_values(["movie", "cell_ID"])
            .reset_index(drop=True)
        )

print("\nObjects ranked by likely pillar-like shape:")
ranked = track_summary.copy()
ranked["pillar_score"] = (
    (ranked["median_eccentricity"] * 2.0) +
    (ranked["median_aspect_ratio"] * 1.0) -
    (ranked["track_motion_std"] * 0.7)
)
display(
    ranked[[
        "movie", "cell_ID", "n_frames", "track_motion_std",
        "median_eccentricity", "median_aspect_ratio", "median_solidity",
        "pillar_score"
    ]]
    .sort_values(["movie", "pillar_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nAvailable results:")
print("preview_filtered_dfs['v1_conservative']")
print("preview_filtered_dfs['v2_balanced']")
print("preview_filtered_dfs['v3_aggressive']")

Preview summary:


,version,pillar_tracks_removed,rows_removed,rows_remaining,min_track_len,max_motion_std,min_ecc,min_aspect,min_solidity
0,v1_conservative,0,0,300,4,2.0,0.75,1.45,0.95
1,v2_balanced,0,0,300,4,2.5,0.72,1.40,0.95
2,v3_aggressive,2,100,200,3,3.0,0.68,1.35,0.94



v1_conservative
No pillar candidates found.

v2_balanced
No pillar candidates found.

v3_aggressive


,movie,cell_ID,n_frames,track_motion_std,median_eccentricity,median_aspect_ratio,median_solidity
0,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,4,50,2.845347,0.802452,1.675836,0.971537
1,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,5,50,2.824933,0.788986,1.627576,0.970962



Objects ranked by likely pillar-like shape:


,movie,cell_ID,n_frames,track_motion_std,median_eccentricity,median_aspect_ratio,median_solidity,pillar_score
0,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,4,50,2.845347,0.802452,1.675836,0.971537,1.288997
1,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,5,50,2.824933,0.788986,1.627576,0.970962,1.228096
2,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,6,50,3.180372,0.752273,1.517792,0.972180,0.796076
3,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,3,50,3.011720,0.314521,1.053464,0.973455,-0.425698
4,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,1,50,3.593411,0.453954,1.122303,0.971407,-0.485177
5,ch20_URA7_URA8_001_Brightfield_GFP_z01_cropped,2,50,4.233547,0.294070,1.046262,0.965956,-1.329082



Available results:
preview_filtered_dfs['v1_conservative']
preview_filtered_dfs['v2_balanced']
preview_filtered_dfs['v3_aggressive']


In [6]:
# Standalone pillar removal script using the v3_aggressive rule

import os
import numpy as np
import pandas as pd

# Paths
csv_path = os.path.join(output_dir, "cell_tracks_per_frame_with_filaments_and_ids.csv")

# Load input
df = pd.read_csv(csv_path)

# Remove old helper columns if present (? some artefact)
helper_cols = [
    "is_pillar",
    "n_frames",
    "track_motion_std",
    "median_eccentricity",
    "median_aspect_ratio",
    "median_solidity",
]
df = df.drop(columns=[c for c in helper_cols if c in df.columns], errors="ignore")

# Required columns
required_cols = [
    "movie",
    "cell_ID",
    "frame",
    "centroid_y",
    "centroid_x",
    "eccentricity",
    "major_axis_length",
    "minor_axis_length",
    "solidity",
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Clean numeric columns
work = df.copy()
numeric_cols = [
    "cell_ID",
    "frame",
    "centroid_y",
    "centroid_x",
    "eccentricity",
    "major_axis_length",
    "minor_axis_length",
    "solidity",
]
for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")

if "area" in work.columns:
    work["area"] = pd.to_numeric(work["area"], errors="coerce")

work = work.dropna(subset=["cell_ID"]).copy()
work["cell_ID"] = work["cell_ID"].astype(int)

work["aspect_ratio"] = (
    work["major_axis_length"] / work["minor_axis_length"].replace(0, np.nan)
)

# Per-track summary
agg_dict = {
    "frame": "nunique",
    "centroid_y": ["std", "median"],
    "centroid_x": ["std", "median"],
    "eccentricity": "median",
    "solidity": "median",
    "aspect_ratio": "median",
}
if "area" in work.columns:
    agg_dict["area"] = "median"

track_summary = work.groupby(["movie", "cell_ID"]).agg(agg_dict)
track_summary.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c
    for c in track_summary.columns
]
track_summary = track_summary.reset_index().rename(columns={
    "frame_nunique": "n_frames",
    "centroid_y_median": "median_centroid_y",
    "centroid_x_median": "median_centroid_x",
    "eccentricity_median": "median_eccentricity",
    "solidity_median": "median_solidity",
    "aspect_ratio_median": "median_aspect_ratio",
    "area_median": "median_area",
})

track_summary["centroid_y_std"] = track_summary["centroid_y_std"].fillna(0)
track_summary["centroid_x_std"] = track_summary["centroid_x_std"].fillna(0)
track_summary["track_motion_std"] = np.sqrt(
    track_summary["centroid_y_std"]**2 + track_summary["centroid_x_std"]**2
)

# v3_aggressive thresholds
MIN_TRACK_LEN = 3
MAX_MOTION_STD = 3.0
MIN_ECC = 0.68
MIN_ASPECT = 1.35
MIN_SOLIDITY = 0.94

track_summary["is_pillar"] = (
    (track_summary["n_frames"] >= MIN_TRACK_LEN) &
    (track_summary["track_motion_std"] <= MAX_MOTION_STD) &
    (track_summary["median_eccentricity"] >= MIN_ECC) &
    (track_summary["median_aspect_ratio"] >= MIN_ASPECT) &
    (track_summary["median_solidity"] >= MIN_SOLIDITY)
)

# Merge labels back to rows
df_labeled = df.merge(
    track_summary[[
        "movie",
        "cell_ID",
        "is_pillar",
        "n_frames",
        "track_motion_std",
        "median_eccentricity",
        "median_aspect_ratio",
        "median_solidity",
    ]],
    on=["movie", "cell_ID"],
    how="left"
)

df_labeled["is_pillar"] = df_labeled["is_pillar"].fillna(False)

# Remove pillars
filtered_df = df_labeled.loc[~df_labeled["is_pillar"]].copy()

# Optional: drop helper columns before final save
helper_to_drop = [
    "is_pillar",
    "n_frames",
    "track_motion_std",
    "median_eccentricity",
    "median_aspect_ratio",
    "median_solidity",
]
filtered_df = filtered_df.drop(
    columns=[c for c in helper_to_drop if c in filtered_df.columns],
    errors="ignore"
)

# Save outputs
filtered_df.to_csv(csv_path, index=False)

removed_tracks_path = os.path.join(output_dir, "removed_pillar_tracks_v3.csv")
track_summary.loc[track_summary["is_pillar"]].to_csv(removed_tracks_path, index=False)

print(f"Saved filtered CSV to: {csv_path}")
print(f"Saved removed pillar track summary to: {removed_tracks_path}")
print(f"Pillar tracks removed: {int(track_summary['is_pillar'].sum())}")
print(f"Rows removed: {int(df_labeled['is_pillar'].sum())}")
print(f"Rows remaining: {len(filtered_df)}")

Saved filtered CSV to: ../data/final_test/results/cell_tracks_per_frame_with_filaments_and_ids.csv
Saved removed pillar track summary to: ../data/final_test/results/removed_pillar_tracks_v3.csv
Pillar tracks removed: 2
Rows removed: 100
Rows remaining: 200
